# Federated junction query and same-record evidence drilldown

This notebook first queries one exact junction across a collection. It then opens one routed source archive and evaluates named junction, region, and terminal-tail predicates on the same locus-resolved archived UMI record. A negative term means only **not observed in that retained evidence unit**, not biological absence. Its final figures are deterministic SVGs built from typed result tables using only the Python standard library and the IPython display API supplied by Colab.

In [ ]:
#@title Immutable demo manifest (required)
MANIFEST_URL = "" #@param {type:"string"}
MANIFEST_SHA256 = "" #@param {type:"string"}
if not MANIFEST_URL or not MANIFEST_SHA256:
    raise RuntimeError("Demo capsule not published/configured: set the immutable manifest URL and its SHA-256. No fallback URL is used.")

In [ ]:
import hashlib, json, re, subprocess, sys, tarfile, urllib.parse, urllib.request, zipfile
from pathlib import Path
WORK = Path('/content/gravlax-demo'); WORK.mkdir(parents=True, exist_ok=True)
HEX64 = re.compile(r'^[0-9a-f]{64}$')
def download_verified(url, sha256, destination):
    parsed_url = urllib.parse.urlsplit(url) if isinstance(url, str) else None
    if parsed_url is None or parsed_url.scheme != 'https' or not parsed_url.netloc: raise ValueError(f'asset URL must be HTTPS, got {url!r}')
    if 'latest' in (segment.lower() for segment in parsed_url.path.split('/')): raise ValueError(f'asset URL must be immutable; /latest/ is not allowed: {url!r}')
    if not isinstance(sha256, str) or not HEX64.fullmatch(sha256): raise ValueError('asset SHA-256 must be 64 lowercase hexadecimal characters')
    destination = Path(destination); temporary = destination.with_suffix(destination.suffix + '.part'); digest = hashlib.sha256()
    with urllib.request.urlopen(url) as source, temporary.open('wb') as sink:
        while block := source.read(1 << 20): digest.update(block); sink.write(block)
    if digest.hexdigest() != sha256:
        temporary.unlink(missing_ok=True); raise RuntimeError(f'SHA-256 mismatch for {url}')
    temporary.replace(destination); return destination
manifest_path = download_verified(MANIFEST_URL, MANIFEST_SHA256, WORK / 'manifest.json')
manifest = json.loads(manifest_path.read_text())
if manifest.get('schema') != 'gravlax.demo-capsule.v1': raise RuntimeError('unsupported or missing demo manifest schema')
for section in ('software', 'resources', 'stories'):
    if not isinstance(manifest.get(section), dict): raise RuntimeError(f'manifest {section} must be an object')
required_software_fields = {'version', 'aie', 'python_wheel'}
if required_software_fields.difference(manifest['software']): raise RuntimeError(f'manifest software lacks {sorted(required_software_fields.difference(manifest["software"]))}')
RESERVED_ASSET_FILENAMES = {'.', '..', 'manifest.json', 'event-discovery.aicollection', 'junction-drilldown.aicollection'}
asset_filenames = {}
for asset_name, asset_spec in [('software.aie', manifest['software'].get('aie')), ('software.python_wheel', manifest['software'].get('python_wheel')), *[(f'resources.{name}', spec) for name, spec in manifest['resources'].items()]]:
    if not isinstance(asset_spec, dict): raise RuntimeError(f'asset declaration {asset_name} must be an object')
    filename = asset_spec.get('filename')
    if not isinstance(filename, str) or not filename or Path(filename).name != filename or filename in RESERVED_ASSET_FILENAMES: raise ValueError(f'asset declaration {asset_name} has an invalid or reserved filename: {filename!r}')
    if filename in asset_filenames: raise ValueError(f'duplicate asset filename {filename!r}: {asset_filenames[filename]} and {asset_name}')
    asset_filenames[filename] = asset_name
def fetch(spec):
    if not isinstance(spec, dict) or any(not spec.get(key) for key in ('url','sha256','filename')): raise RuntimeError(f'incomplete published asset declaration: {spec!r}')
    if Path(spec['filename']).name != spec['filename']: raise ValueError('asset filename must be a basename')
    return download_verified(spec['url'], spec['sha256'], WORK / spec['filename'])
def install_tools():
    wheel = fetch(manifest['software']['python_wheel']); subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--force-reinstall', '--no-deps', str(wheel)], check=True)
    spec = manifest['software']['aie']; bundle = fetch(spec); member = spec.get('member')
    if member:
        if zipfile.is_zipfile(bundle):
            with zipfile.ZipFile(bundle) as archive: payload = archive.read(member)
        else:
            with tarfile.open(bundle, 'r:*') as archive:
                item = archive.getmember(member)
                if not item.isfile(): raise RuntimeError('configured aie archive member is not a file')
                payload = archive.extractfile(item).read()
        binary = WORK / 'aie'; binary.write_bytes(payload)
    else: binary = bundle
    binary.chmod(0o755); return binary
AIE = install_tools()
from gravlax import Client, __version__ as PYTHON_VERSION
EXPECTED_VERSION = manifest['software'].get('version')
if not isinstance(EXPECTED_VERSION, str) or not EXPECTED_VERSION: raise RuntimeError('manifest software.version must be nonempty')
CLI_VERSION = subprocess.run([str(AIE), '--version'], check=True, capture_output=True, text=True).stdout.strip()
if CLI_VERSION != f'aie {EXPECTED_VERSION}': raise RuntimeError(f'CLI version mismatch: {CLI_VERSION!r} != aie {EXPECTED_VERSION}')
if PYTHON_VERSION != EXPECTED_VERSION: raise RuntimeError(f'Python version mismatch: {PYTHON_VERSION!r} != {EXPECTED_VERSION}')
client = Client(binary=AIE); print(CLI_VERSION)

In [ ]:
story = manifest['stories'].get('junction_drilldown'); resources = manifest['resources']
required_story_fields = {'archives', 'archive_sample', 'junction', 'predicates', 'expression', 'universe', 'emit_membership', 'expected_min_selected_units', 'required_true_predicates', 'story_note'}
if not isinstance(story, dict) or required_story_fields.difference(story): raise RuntimeError(f'junction_drilldown story lacks {sorted(required_story_fields.difference(story or {}))}')
if not isinstance(story['archives'], dict) or not story['archives']: raise RuntimeError('junction_drilldown.archives must be a nonempty sample-to-resource object')
if story['archive_sample'] not in story['archives']: raise RuntimeError('archive_sample is not present in junction_drilldown.archives')
if not isinstance(story['predicates'], dict) or not story['predicates']: raise RuntimeError('junction_drilldown.predicates must be a nonempty object')
if story['emit_membership'] is not True: raise RuntimeError('the bounded drilldown contract requires emit_membership=true')
if not isinstance(story['story_note'], str) or not story['story_note'].strip(): raise RuntimeError('junction_drilldown.story_note must be nonempty')
print('Interpretation:', story['story_note'])
def story_resource(name):
    spec = resources.get(name)
    if not isinstance(name, str) or not isinstance(spec, dict): raise RuntimeError(f'story resource is not declared: {name!r}')
    return spec
archives = {sample: fetch(story_resource(resource)) for sample, resource in story['archives'].items()}
for sample, path in archives.items():
    expected = story_resource(story['archives'][sample]).get('archive_root')
    if not expected: raise RuntimeError(f'archive {sample} lacks a rooted identity in the manifest')
    identity = client.result_raw(['inspect-archive', path, '--json'])['native_identity']
    observed = f"{identity['scheme']}:{identity['blake3']}"
    if observed != expected: raise RuntimeError(f'archive root mismatch for {sample}: {observed}')
collection = WORK / 'junction-drilldown.aicollection'; collection.unlink(missing_ok=True)
build = ['collection', 'build']
for sample, path in sorted(archives.items()): build.append(f'--sample={sample}={path}')
if story.get('shape_routes', True): build.append('--shape-routes')
if story.get('allow_unstamped', False): build.append('--allow-unstamped')
build.extend([f'--out={collection}']); client.run(build)
archive = archives[story['archive_sample']]
junction = story['junction']
federated = client.result_bundle(['collection', 'junction', collection, junction, '--top=0', '--format=json'])
print(json.dumps(federated.summary.as_dict(), indent=2))
for sample in federated.table('samples').records(): print(sample)

In [ ]:
cooccurrence = client.query_cooccurrence(
    archive, story['predicates'], story['expression'],
    universe=story['universe'],
    unit=story.get('unit', 'molecule-record'),
    region_match=story.get('region_match', 'anchor'),
    placements=story.get('placements', 'unique'),
    allow_full_scan=story.get('allow_full_scan', False),
    groups=(fetch(story_resource(story['drilldown_groups'])) if story.get('drilldown_groups') else None),
    aggregation=story.get('aggregation', 'cell'),
    emit_membership=story['emit_membership'],
    max_memberships=story.get('max_memberships', 1000000),
    max_pattern_rows=story.get('max_pattern_rows', 1000000),
    max_chunks=story.get('max_chunks', 100000),
    max_evidence_records=story.get('max_evidence_records', 10000000),
    max_terminal_events=story.get('max_terminal_events', 10000000),
)
cooccurrence_summary = cooccurrence.summary.as_dict()
print(json.dumps(cooccurrence_summary, indent=2))
required_result_tables = {'predicates', 'patterns', 'memberships'}
if required_result_tables.difference(cooccurrence.table_names): raise RuntimeError(f'co-occurrence result lacks {sorted(required_result_tables.difference(cooccurrence.table_names))}')
if cooccurrence_summary.get('selected_units', 0) < story['expected_min_selected_units']: raise RuntimeError('selected-unit count fell below its locked minimum')
patterns = cooccurrence.table('patterns').records()
membership_table = cooccurrence.table('memberships')
membership_columns = {'cell_id', 'unit_id', 'barcode', 'umi_class', 'chunk', 'local_record', 'global_record', 'contributing_records', 'pattern_mask', 'completeness_mask', 'matched_predicates', 'selection_state', 'selected'}
if membership_columns.difference(membership_table.columns): raise RuntimeError(f'memberships table lacks {sorted(membership_columns.difference(membership_table.columns))}')
memberships = membership_table.records()
if not memberships: raise RuntimeError('co-occurrence result emits no membership witnesses')
required_true = set(story['required_true_predicates'])
if not any(row.get('selection_state') == 'true' and row.get('evidence_units', 0) > 0 and required_true.issubset(row.get('matched_predicates', ())) for row in patterns): raise RuntimeError('required true predicate pattern is absent')
selected_memberships = [row for row in memberships if row.get('selection_state') == 'true' and row.get('selected') is True]
if len(selected_memberships) != cooccurrence_summary['selected_units']: raise RuntimeError('selected-unit summary differs from membership witnesses')
if not any(required_true.issubset(row.get('matched_predicates', ())) for row in selected_memberships): raise RuntimeError('selected memberships lack the required predicate witness')
for pattern in patterns: print(pattern)
for unit in memberships[:20]: print(unit)

In [ ]:
# Deterministic SVGs generated only from the typed junction and pattern tables.
from html import escape
from IPython.display import SVG, display

def require_table_columns(table, expected, label):
    missing = set(expected).difference(table.columns)
    if missing: raise RuntimeError(f'{label} table is missing columns: {sorted(missing)}')

def support_bar_svg(items, title, value_label, empty_message):
    width, label_width, plot_width, row_height = 940, 350, 480, 28
    items = items[:16]; height = 92 + row_height * max(1, len(items))
    peak = max((value for _, value, _ in items), default=1) or 1
    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" role="img" aria-label="{escape(title)}">',
             '<style>text{font-family:system-ui,sans-serif;font-size:12px}.title{font-size:16px;font-weight:600}.axis{fill:#5b6470}</style>',
             f'<text class="title" x="8" y="22">{escape(title)}</text>',
             f'<text class="axis" x="{label_width}" y="43">{escape(value_label)}</text>']
    if not items: parts.append(f'<text x="8" y="72">{escape(empty_message)}</text>')
    for index, (label, value, color) in enumerate(items):
        y = 58 + index * row_height; span = value * plot_width / peak
        parts.extend([f'<text x="{label_width - 8}" y="{y + 14}" text-anchor="end">{escape(str(label)[:52])}</text>',
                      f'<rect x="{label_width}" y="{y}" width="{max(span, 1)}" height="18" rx="2" fill="{color}"/>',
                      f'<text x="{label_width + span + 5}" y="{y + 14}">{value}</text>'])
    parts.append('</svg>'); return ''.join(parts)

sample_table, pattern_table = federated.table('samples'), cooccurrence.table('patterns')
require_table_columns(sample_table, {'sample', 'present', 'umis'}, 'junction samples')
require_table_columns(pattern_table, {'entity', 'matched_predicates', 'selection_state', 'evidence_units'}, 'co-occurrence patterns')
junction_rows = sorted(((row['sample'], row['umis'], '#3366cc') for row in sample_table.records()), key=lambda item: (-item[1], item[0]))
pattern_rows = []
state_colors = {'true': '#177e89', 'false': '#9aa0a6', 'unknown': '#e6a700'}
for row in pattern_table.records():
    names = row['matched_predicates']
    if not isinstance(names, list) or not all(isinstance(name, str) for name in names): raise RuntimeError('patterns.matched_predicates must be a string array')
    label = f"{row['entity']} · {','.join(names) or 'none'} [{row['selection_state']}]"
    pattern_rows.append((label, row['evidence_units'], state_colors.get(row['selection_state'], '#9aa0a6')))
pattern_rows.sort(key=lambda item: (-item[1], item[0]))
display(SVG(data=support_bar_svg(junction_rows, f'Exact junction support · {junction}', 'UMIs by source archive', 'No source archive supports this junction.')))
display(SVG(data=support_bar_svg(pattern_rows, 'Same-record evidence patterns', 'locus-resolved archived UMI records', 'No evidence unit matched the universe predicate.')))